# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.19.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

!pip install -q llmcompressor==0.10.0.2
!pip install -q compressed-tensors==0.14.0.1

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 106.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 30.4 MB/s eta 0:00:0000:01:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-22 00:15:54.892727: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779408955.100596      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779408955.161252      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779408955.650505      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779408955.650550      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779408955.650553      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Phi-4-mini-instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Phi-4-mini-instruct-WANDA-AWQ-4bit"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"

MODEL_NAME           = "Phi-4-mini-instruct"
MODEL_NAME_IN_REPO   = "Phi-4-mini-instruct-WANDA-AWQ"
COMPRESSION_METHOD   = "WANDA & AWQ"
MODEL_TYPE           = "Pruning & Quantization"
SPARSITY = 25
PATH = f"Sparsity/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

recipe.yaml:   0%|          | 0.00/730 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Sparsity/25/model.safetensors:   0%|          | 0.00/2.89G [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Sparsity/25/tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [10]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=f"{local_path}/{PATH}",
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-22 00:16:42 [utils.py:233] non-default args: {'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': '/kaggle/working/model/Sparsity/25'}
INFO 05-22 00:16:42 [config.py:446] Replacing legacy 'type' key with 'rope_type'
INFO 05-22 00:17:04 [model.py:549] Resolved architecture: Phi3ForCausalLM
WARNING 05-22 00:17:04 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-22 00:17:04 [model.py:1678] Using max model len 256
INFO 05-22 00:17:05 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-22 00:17:05 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-22 00:17:06 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-

2026-05-22 00:17:19.528701: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779409039.556897     767 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779409039.565426     767 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779409039.585807     767 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779409039.585840     767 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779409039.585843     767 computation_placer.cc:177] computation placer alr

(EngineCore pid=767) INFO 05-22 00:17:28 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='/kaggle/working/model/Sparsity/25', speculative_config=None, tokenizer='microsoft/Phi-4-mini-instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=256, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=compressed-tensors, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_e

2026-05-22 00:17:33.407577: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-22 00:17:33.424962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779409053.436514     792 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779409053.444690     792 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1779409053.456813     793 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
W0000 00:00:1779409053.463596     792 computation_pl

(Worker pid=792) INFO 05-22 00:17:47 [parallel_state.py:1400] world_size=2 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:57753 backend=nccl
(Worker pid=793) INFO 05-22 00:17:47 [parallel_state.py:1400] world_size=2 rank=1 local_rank=1 distributed_init_method=tcp://127.0.0.1:57753 backend=nccl


...

(Worker_TP0 pid=792) INFO 05-22 00:17:51 [default_loader.py:384] Loading weights took 1.64 seconds
(Worker_TP0 pid=792) INFO 05-22 00:17:52 [gpu_model_runner.py:4820] Model loading took 1.37 GiB memory and 2.408617 seconds
(Worker_TP0 pid=792) INFO 05-22 00:18:06 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/653454d26f/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=792) INFO 05-22 00:18:06 [backends.py:1111] Dynamo bytecode transform time: 12.70 s
(Worker_TP0 pid=792) INFO 05-22 00:18:13 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=792) INFO 05-22 00:18:20 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 12.66 s
(Worker_TP0 pid=792) INFO 05-22 00:18:23 [decorators.py:640] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/704cfc15fab5469f4b7fc649305c6d87e6888c8dd3dcd3c4dd41ed0082f067aa/rank_0_0/model
(Worker_TP0 pid=792) INFO 05-22 00:18:2

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.38it/s]
Capturing CUDA graphs (decode, FULL):  97%|█████████▋| 34/35 [00:05<00:00,  3.58it/s]

(Worker_TP1 pid=793) INFO 05-22 00:18:55 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses
(Worker_TP0 pid=792) INFO 05-22 00:18:55 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.13it/s]


(Worker_TP1 pid=793) INFO 05-22 00:18:55 [gpu_worker.py:597] CUDA graph pool memory: 0.47 GiB (actual), 0.49 GiB (estimated), difference: 0.02 GiB (3.7%).
(Worker_TP0 pid=792) INFO 05-22 00:18:56 [gpu_model_runner.py:6046] Graph capturing finished in 13 secs, took 0.47 GiB
(Worker_TP0 pid=792) INFO 05-22 00:18:56 [gpu_worker.py:597] CUDA graph pool memory: 0.47 GiB (actual), 0.49 GiB (estimated), difference: 0.02 GiB (3.7%).
(EngineCore pid=767) INFO 05-22 00:18:56 [core.py:283] init engine (profile, create kv cache, warmup model) took 63.19 seconds
(EngineCore pid=767) INFO 05-22 00:18:57 [vllm.py:790] Asynchronous scheduling is enabled.
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times    = []
latency_times = []

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    # Extract from RequestStateStats fields
    ttft    = metrics.first_token_latency                    # prefill cost
    latency = metrics.last_token_ts - metrics.queued_ts      # total e2e latency

    ttft_times.append(ttft)
    latency_times.append(latency)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr,     latency_arr    = np.array(ttft_times), np.array(latency_times)

ttft_mean,    ttft_std       = ttft_arr.mean(),    ttft_arr.std()
latency_mean, latency_std    = latency_arr.mean(), latency_arr.std()

# decode throughput: excludes first token, over decode-only window
throughput_arr               = (MAX_GENERATION_TOKENS - 1) / (latency_arr - ttft_arr)
throughput_mean, throughput_std = throughput_arr.mean(), throughput_arr.std()

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-22 00:19:09 [loggers.py:259] Engine 000: Avg prompt throughput: 17.1 tokens/s, Avg generation throughput: 88.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-22 00:19:19 [loggers.py:259] Engine 000: Avg prompt throughput: 13.3 tokens/s, Avg generation throughput: 113.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 00:19:29 [loggers.py:259] Engine 000: Avg prompt throughput: 15.2 tokens/s, Avg generation throughput: 111.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-22 00:19:39 [loggers.py:259] Engine 000: Avg prompt throughput: 13.3 tokens/s, Avg generation throughput: 109.7 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%


In [13]:
print(latency_times)

[1.3031799369999817, 1.3081042329999946, 1.3075361520000115, 1.3092155360000106, 1.3117112920000409, 1.319428194000011, 1.3177168220000794, 1.3197599900000796, 1.3266626619999897, 1.3257918130000235, 1.329108609000059, 1.3313889330000848, 1.3325942719999375, 1.336167445000001, 1.3349462529999983, 1.3411511010000368, 1.3469228189999285, 1.3528347760000088, 1.347382808999896, 1.3567610820000482, 1.35482097900001, 1.356046718000016, 1.3663582819999647, 1.3603317609999976, 1.367881744999977, 1.3807619309999382, 1.3690807630000563, 1.3724722050000082, 1.3793199940000704, 1.3763348160000533]


In [14]:
print(ttft_times)

[0.02935004234313965, 0.0335085391998291, 0.027051448822021484, 0.02835392951965332, 0.027756214141845703, 0.030356884002685547, 0.026231050491333008, 0.026729106903076172, 0.028764724731445312, 0.032755374908447266, 0.033899784088134766, 0.030338764190673828, 0.027507543563842773, 0.030516862869262695, 0.027076244354248047, 0.027424097061157227, 0.027502775192260742, 0.03365635871887207, 0.027441978454589844, 0.03188347816467285, 0.02757096290588379, 0.028047800064086914, 0.03007984161376953, 0.02910780906677246, 0.036875247955322266, 0.04378056526184082, 0.028531551361083984, 0.028432130813598633, 0.03004908561706543, 0.027980804443359375]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"            : MODEL_NAME,
    "model_type"            : MODEL_TYPE,
    "compression_method"    : COMPRESSION_METHOD,
    "sparsity"              : SPARSITY,
    "input_token_count"     : input_tokens,
    "generated_token_count" : total_generated_tokens,
    "ttft_sec"              : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "latency_sec"           : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "throughput_tokens_per_sec" : f"{throughput_mean:.2f} ± {throughput_std:.2f}",
}


# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [16]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}/Sparsity/{SPARSITY}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main 1a6a7e6] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-WANDA-AWQ/Sparsity/25
 1 file changed, 11 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-WANDA-AWQ/Sparsity/25/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-WANDA-AWQ/Sparsity/25'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   fd76f53..1a6a7e6  main -> main
